# Para la creación de la hipotesis haremos los siguientes pasos:

0. Traer librerias a usar y el df

1. Entendimiento de los datos

2. Entrada X, salida Y 

3. Selección de la hipotesis y Argumentos a favor y en contra

4. Preprocesamiento de datos sin fuga

5. Parametros e hiperparametros a estimar

6. Entrenamiento y evaluacion 

# 0. Librerias a usar y el df

In [1]:
from sklearn import set_config
set_config(display="diagram")

In [2]:
import pandas as pd
import sklearn as sk
import numpy as np
import kagglehub
import os, glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score
)

c:\Users\jeam2\Downloads\maestria de Ciencia de Datos\II-semestre segundo_corte\Mét. Apren. Auto. para Toma De\trabajo entregable\Machine_Learning_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Path to dataset files:", path)

csv = glob.glob(os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))[0]
df = pd.read_csv(csv)
df.head()

Path to dataset files: C:\Users\jeam2\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Rúbrica 4
## El documento reporta la realización de la etapa de entendimiento de los datos para identificar si los supuestos hechos en la hipótesis sobre ellos son reales.

# 1.Entendimiento de los datos

In [4]:
print(df.info())
print("\nDistribución de Churn:")
print(df["Churn"].value_counts(normalize=True))

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

# 2. Entrada X y salida Y

In [5]:
df_ = df.copy()

y = df_["Churn"].map({"Yes": 1, "No": 0})
X = df_.drop(columns=["customerID", "Churn"])

# 3. Selección de la hipotesis
## Rúbrica 1
# El documento reporta la selección de las dos hipótesis más adecuadas

Nuestras hipotesis seleccionadas son las siguientes:

H1: SVM es buena para realizar la clasificacion binaria con frontera y requiere un escalado numerico

H2 Random forest es robusto con los datos tabulares y puede capturar relaciones no lineales

## Rúbrica 2
## El documento reporta argumentos a favor de la selección de cada hipótesis elegida

A favor SVM:

Funciona bien en clasificación binaria con fronteras complejas: en churn, la salida depende de combinaciones (p. ej. Contract + tenure + MonthlyCharges + servicios). Un SVM con kernel Gaussiano permite modelar separaciones no lineales (fronteras “curvas”), no solo una línea/plano.
Adecuado para un dataset de tamaño medio: Telco tiene de 7043 filas, un tamaño donde SVM suele ser viable computacionalmente si se hace bien el preprocesamiento.
Buena opción cuando hay muchas variables categóricas codificadas: después de One-Hot, el espacio se vuelve de alta dimensión; SVM puede rendir bien en estos escenarios si está correctamente escalado.

A favor Random Forest:

Muy fuerte en datos mixtos: Telco combina variables numéricas (tenure, cargos) y muchas categóricas (servicios, contrato, método de pago). Random Forest captura interacciones y no linealidades sin requerir que se definan.
Robusto y estable: al ser un ensamble de muchos árboles, suele generalizar bien y reducir la tendencia al sobreajuste frente a un árbol único.
Explicabilidad para negocio: te permite obtener importancia de variables, útil para entender qué factores empujan el churn (ej. tenure, contract, monthly charges, etc.). Esto ayuda mucho a justificar decisiones de retención.
Menos dependiente del escalado: a diferencia de SVM, Random Forest no necesita obligatoriamente estandarización para funcionar bien (aunque igual se necesita hacer el split y evitar fuga con pipelines).

## Rúbrica 3
## El documento reporta argumentos en contra de las hipótesis no seleccionadas

En contra Perceptrón Multicapa (no seleccionado):

Alta necesidad de ajuste fino: en este tipo de datasets se suele requerir elección cuidadosa de número de capas/neuronas, activaciones, regularización y tasa de aprendizaje; si no se ajusta bien puede sobreajustar o quedarse corto sin mejorar a modelos clásicos.
Sensibilidad al preprocesamiento: necesita escalado de numéricas y una codificación consistente de categóricas (one-hot). Si hay variables como TotalCharges con valores vacíos/strings, el MLP se afecta más si no se limpian bien.
Interpretabilidad limitada: Un MLP es más tipo “caja negra” que Random Forest, lo que dificulta justificar decisiones ante negocio.
Eficiencia vs beneficio: con aprox. 7k filas, el costo de tuning y validación del MLP puede ser mayor que el beneficio, y muchas veces no supera a ensambles (RF) en este tipo de dataset tabular.

En contra K-Vecinos (KNN):

La distancia pierde sentido con One-Hot: Telco tiene muchas categorías; al convertirlas a one-hot, se generan muchas dimensiones y la noción de “vecinos cercanos” se vuelve menos informativa.
Muy sensible a la escala: si MonthlyCharges y tenure no están escaladas, dominan la distancia; si las escalas cambian, el rendimiento cambia mucho. Eso hace a KNN frágil si el preprocesamiento no es perfecto.
Inferencia costosa: KNN predice comparando contra muchos puntos del entrenamiento; aunque el dataset no es enorme, es menos eficiente que modelos que aprenden una función (SVM/RF).
Puede sesgarse por desbalance: si hay más “No Churn” que “Churn”, KNN tiende a votar por la clase mayoritaria a menos que ajustes pesos/vecinos y métricas de distancia con cuidado.


# 4. Preprocesamiento de datos sin fuga


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Convertidor seguro de TotalCharges (dentro del pipeline)
class FixTotalCharges(BaseEstimator, TransformerMixin):
    def __init__(self, col="TotalCharges"):
        self.col = col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Convierte strings con espacios a numérico; lo no convertible -> NaN
        X[self.col] = pd.to_numeric(X[self.col].astype(str).str.strip(), errors="coerce")
        return X

# Columnas numéricas esperadas (incluye TotalCharges ya convertido por FixTotalCharges)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())     # crítico para SVM
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

# 5. Parametros e hiperparametros a estimar



In [7]:
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

svm_pipe = Pipeline(steps=[
    ("fix_totalcharges", FixTotalCharges("TotalCharges")),
    ("prep", preprocess),
    ("clf", svm_model)
])

rf_pipe = Pipeline(steps=[
    ("fix_totalcharges", FixTotalCharges("TotalCharges")),
    ("prep", preprocess),
    ("clf", rf_model)
])

## Rúbrica 6

# 6. Entrenamiento y evaluacion 

In [8]:
def evaluate_model(pipe, X_test, y_test, name="model"):
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None

    print(f"\n==================== {name} ====================")
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print("\nClassification report:\n", classification_report(y_test, pred))
    if proba is not None:
        print("ROC-AUC:", roc_auc_score(y_test, proba))

# Entrenar SOLO con train
svm_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)

# Evaluar en test (sin fuga)
evaluate_model(svm_pipe, X_test, y_test, name="SVM (RBF)")
evaluate_model(rf_pipe, X_test, y_test, name="Random Forest")



==================== SVM (RBF) ====================
Confusion matrix:
 [[1135  417]
 [ 122  439]]

Classification report:
               precision    recall  f1-score   support

           0       0.90      0.73      0.81      1552
           1       0.51      0.78      0.62       561

    accuracy                           0.74      2113
   macro avg       0.71      0.76      0.71      2113
weighted avg       0.80      0.74      0.76      2113

ROC-AUC: 0.8206701260635464

==================== Random Forest ====================
Confusion matrix:
 [[1386  166]
 [ 292  269]]

Classification report:
               precision    recall  f1-score   support

           0       0.83      0.89      0.86      1552
           1       0.62      0.48      0.54       561

    accuracy                           0.78      2113
   macro avg       0.72      0.69      0.70      2113
weighted avg       0.77      0.78      0.77      2113

ROC-AUC: 0.8168196519470018


# 7. Validación repetida

In [9]:
sss = StratifiedShuffleSplit(n_splits=10, test_size=0.30, random_state=42)

def cv_auc(pipe, X, y, splitter, name="model"):
    aucs = []
    for tr, te in splitter.split(X, y):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        m = Pipeline(steps=pipe.steps)  # copia ligera del pipeline
        m.fit(X_tr, y_tr)
        proba = m.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, proba))

    print(f"\nAUC promedio ({name}): {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")